# L.I.V.E. — síntese de voz no Colab

Gera falas de números com motores de licença comercial clara. **Só dados sintéticos ou públicos aqui** (Princípio IV): a gravação real do PO nunca vem para a nuvem.

O Colab usa Python 3.13, e o Kokoro recente exige < 3.13 (com 3.13 o pip cai no Kokoro 0.7, sem vozes pt-BR, e tenta compilar numpy). Por isso tudo roda num ambiente **Python 3.12** criado com `uv`, igual ao da máquina local.

In [ ]:
BRANCH = "002-modelo-ctc-onnx"
ENGINE = "kokoro"  # kokoro | chatterbox | parler
!git clone --depth 1 -b {BRANCH} https://github.com/yuriwinchest/live-engine-training.git
%cd live-engine-training/training
!apt-get -qq -y install espeak-ng > /dev/null
!pip install -q uv && uv venv -q -p 3.12 /content/venv312
V = "/content/venv312/bin/python"
EXTRA = {"kokoro": "'kokoro>=0.9.4' 'transformers>=4.40'", "chatterbox": "chatterbox-tts", "parler": "git+https://github.com/huggingface/parler-tts.git"}[ENGINE]
# --python explícito: o Colab define UV_SYSTEM_PYTHON=true e o uv instalaria no Python do sistema.
!uv pip install -q --python {V} -e . {EXTRA}
!{V} -c "import torch; print('GPU:', torch.cuda.is_available())"

Chatterbox clona a voz de clipes de referência **CC0** (Common Voice pt). Envie a pasta para `/content/refs`.

In [ ]:
COUNT, SEED = 3000, 7
OPTS = "" if ENGINE != "chatterbox" else "--engine-option references=/content/refs"
L = f"{V} -m live_lab"
!{L} synth plan --engine {ENGINE} {OPTS} --out data/jobs-{ENGINE}.jsonl --count {COUNT} --seed {SEED}
!{L} synth render --engine {ENGINE} {OPTS} --jobs data/jobs-{ENGINE}.jsonl --out data/clean/{ENGINE}

Se a sessão cair, rode as células de novo: a renderização continua de onde parou (dentro da mesma sessão).

## Levar o resultado

In [ ]:
!cd data/clean && zip -qr /content/clean-{ENGINE}.zip {ENGINE}
from google.colab import files
files.download(f"/content/clean-{ENGINE}.zip")